# VAE code

In [ ]:
import torch
import torchvision.datasets as datasets
import torchvision.transforms as transforms
import numpy as np
import torch.optim as optim
import torch.nn.functional as F
import torch.nn as nn                                   ### alleged missing packages, from chatGPT
from torch.utils.data import DataLoader, TensorDataset  ### ^^^

class VAE(nn.Module):   ### parameters of class declarations are other classes. ref example @ bottom.
                        ### so VAE doesn't take in a variable (object), it takes in the CLASS from
                        ### pytorch, nn.Module. This declaration is how super().__init__() knows the relation
                        ### to the actual pytorch thing we need to "inherit" from
    def __init__(
          self, 
          x_dim,
          hidden_dim,
          z_dim=10
        ):
        super(VAE, self).__init__() ### yes, this formatting is correct
                                    ### init only takes args if PARENT class takes them

        # Define autoencoding layers
        self.enc_layer1 = nn.Linear(x_dim, hidden_dim)
        self.enc_layer2_mu = nn.Linear(hidden_dim, z_dim)       ### DIFF: just splits the second layer, hidden_dim -> z_dim, into
        self.enc_layer2_logvar = nn.Linear(hidden_dim, z_dim)   ### TWO parts, one for mu and one for logvar, instead of ONE simple one:
                                                                ### input -> hidden -> (mu, logvar), vs. input -> hidden -> Z <<< SIMPLE Z -
                                                                ### only sampling -- no dist's
        ### one last note -- why logvar instead of just var? var >= 0, but log maps all num's > 0 to pos and neg num's -- more flexible
        
        # Define autoencoding layers
        self.dec_layer1 = nn.Linear(z_dim, hidden_dim)
        self.dec_layer2 = nn.Linear(hidden_dim, x_dim) 

    def encoder(self, x):
        x = F.relu(self.enc_layer1(x))
        mu = F.relu(self.enc_layer2_mu(x))
        logvar = F.relu(self.enc_layer2_logvar(x))
        return mu, logvar                           ### again, instead of getting Z after encoding, we get parameters mu and logvar to create probabilistic Z

    def reparameterize(self, mu, logvar):
        std = torch.exp(logvar/2)
        eps = torch.randn_like(std)
        z = mu + std * eps
        return z                                ### don't fully understand, but we reparameterize because otherwise, it would affect backpropagation during training

    def decoder(self, z):
        # Define decoder network                    ### ChatGPT recommends sigmoid instead of relu for images. Does it matter?
        output = F.relu(self.dec_layer1(z))
        output = F.relu(self.dec_layer2(output))
        return output

    def forward(self, x):                       ### forward() not called in code, because it is called through nn.Module
        mu, logvar = self.encoder(x)            ### nn.Module just req's you to make your own forward()
        z = self.reparameterize(mu, logvar)
        output = self.decoder(z)
        return output, z, mu, logvar


def train_model(
    X, 
    learning_rate=1e-4, 
    batch_size=128, 
    num_epochs=15,
    hidden_dim=256,
    latent_dim=50
  ):
  # Convert X to a PyTorch tensor
  X = torch.tensor(X).float()
  # Create DataLoader object to generate minibatches
  dataset = TensorDataset(X)
  dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)
  
  # Define the VAE model
  model = VAE(x_dim=X.shape[1], hidden_dim=hidden_dim, z_dim=latent_dim)
  # Define the optimizer
  optimizer = optim.Adam(model.parameters(), lr=learning_rate)
    
  # Define the loss function
  def loss_function(output, x, mu, logvar):
        recon_loss = F.mse_loss(output, x, reduction='sum') / batch_size    ### DIFF: div by batch size. idk why AE doesn't
        kl_loss = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())   ### DIFF: KL divergence. Not present in AE
        return recon_loss + 0.002  * kl_loss                                ### the KL being present is exactly what diff'tes VAE from AE. it arises because we are trying to construct standard normal dist's, and req KL loss to guide that

  # Train the model
  for epoch in range(num_epochs):
      epoch_loss = 0
      for batch in dataloader:
          # Zero the gradients
          optimizer.zero_grad()

          # Get batch
          x = batch[0]

          # Forward pass
          output, z, mu, logvar = model(x)

          # Calculate loss
          loss = loss_function(output, x, mu, logvar)

          # Backward pass
          loss.backward()

          # Update parameters
          optimizer.step()

          # Add batch loss to epoch loss
          epoch_loss += loss.item()

      # Print epoch loss
      print(f"Epoch {epoch+1}/{num_epochs}, Loss: {epoch_loss/len(X)}")
      
  return model


### differences b/w VAE, AE

- b/c VAE is essentially a probablistic AE, the latent is "endowed" with a probability distribution, ie normal distribution, and has parameters mu and logvar. Big difference is that these parameters come up very often in our creation of the VAE, and so there are additional considerations to fit these parameters/the probabilistic nature back into the VAE
- 

# testing random code stuff

In [ ]:
class Parent:
    def __init__(self, a):
        self.a = a

class Child(Parent):
    def __init__(self, a, b):
        super().__init__(a)
        self.b = b

ent = Child("B", "A")
print(ent.a, ent.b)

Alice 123
